In [ ]:
import os
import shutil
import yaml
import bagpy
from bagpy import bagreader
import pandas as pd
import numpy as np
from scipy.integrate import solve_ivp
from tkinter import Tk     # from tkinter import Tk for Python 3.x
from tkinter.filedialog import askopenfilename
from MPCModelFuncs import differential_mpc_model, planar_model_TV
from MPCModelFuncs import euler_step
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import rosbag
from NN_models import Model18, Model20
import pandas as pd

# modelType = 'model_18'
h = 0.05

# Read yaml file
p = np.zeros(24)
with open('/home/david/fst/autonomous-systems/src/control/learning_mpcc/config/default.yaml') as stream:
    parameters = yaml.load(stream, Loader=yaml.SafeLoader)
    p[0] = parameters['model_params']['l_f']
    p[1] = parameters['model_params']['l_r']
    p[2] = parameters['model_params']['m']
    p[3] = parameters['model_params']['I_z']
    p[4] = parameters['model_params']['T_max_front']
    p[5] = parameters['model_params']['T_max_rear']
    p[6] = parameters['model_params']['T_brake_front']
    p[7] = parameters['model_params']['T_brake_rear']
    p[8] = parameters['model_params']['GR']
    p[9] = parameters['model_params']['eta_motor']
    p[10] = parameters['model_params']['r_wheel']
    p[11] = parameters['model_params']['g']
    p[12] = parameters['model_params']['C_roll']
    p[13] = parameters['model_params']['rho']
    p[14] = parameters['model_params']['C_d']
    p[15] = parameters['model_params']['C_l']
    p[16] = parameters['tyre_params']['B']
    p[17] = parameters['tyre_params']['C']
    p[18] = parameters['tyre_params']['D']
    p[19] = parameters['model_params']['downforce_front']
    p[20] = parameters['model_params']['downforce_rear']
    p[21] = parameters['model_params']['h_cog']
    p[22] = parameters['model_params']['track_width']
    p[23] = parameters['model_params']['diff_gain']
    vel_min = parameters['model_params']['lambda_blend_min']


In [ ]:
# Load Dataset
train_in_dataset = []
train_out_dataset = []
test_in_dataset = []
test_out_dataset = []
bag = rosbag.Bag('/home/david/bags/test_before_october_all/online/2023-09-27-15-30-00.bag')
for topic, msg, t in bag.read_messages():
    if topic == '/control/model_learning/dataset':
        train_in_dataset.append(msg.train_in_dataset)
        train_out_dataset.append(msg.train_out_dataset)
        test_in_dataset.append(msg.test_in_dataset)
        test_out_dataset.append(msg.test_out_dataset)
    if topic == '/control/model_learning/info':
        train_loss = msg.train_loss
        test_loss = msg.test_loss

print(f"Train Loss : {train_loss}; Test Loss : {test_loss}")

train_in_dataset = torch.tensor(train_in_dataset[len(train_in_dataset)-1]).reshape(-1, 2, 5)
train_out_dataset = torch.tensor(train_out_dataset[len(train_out_dataset)-1]).reshape(-1, 3)
test_in_dataset = torch.tensor(test_in_dataset[len(test_in_dataset)-1]).reshape(-1, 2, 5)
test_out_dataset = torch.tensor(test_out_dataset[len(test_out_dataset)-1]).reshape(-1, 3)


# Load Models
models = []
model_path = '/home/david/fst/autonomous-systems/src/control/model_learning/saved_model_cpp/2023-09-27_15-30-00/'
models_names = [f for f in os.listdir(model_path) if f.endswith(".pth")]
for model_name in models_names:
    # if modelType == 'model_18':
    model = torch.jit.load(model_path + model_name)
    model.eval()
    models.append(model)
    print(model_name)

In [ ]:
states_next_model = np.zeros((len(models), len(test_in_dataset), 3))
states_next_NN = np.zeros((len(models), len(test_in_dataset), 3))

test_in_dataset_np = test_in_dataset.numpy()
# States
vx = test_in_dataset_np[:, -1, 2]
vx = vx[:,np.newaxis]
vy = test_in_dataset_np[:, -1, 3]
vy = vy[:,np.newaxis]
r = test_in_dataset_np[:, -1, 4]
r = r[:,np.newaxis]
vx_prev = test_in_dataset_np[:, -2, 2]
vx_prev = vx_prev[:,np.newaxis]
vy_prev = test_in_dataset_np[:, -2, 3]
vy_prev = vy_prev[:,np.newaxis]
states = np.hstack((vx,vy,r))
# Inputs
throttle = test_in_dataset_np[:, -1, 0]
throttle = throttle[:,np.newaxis]
steering = test_in_dataset_np[:, -1, 1]
steering = steering[:,np.newaxis]
throttle_prev = test_in_dataset_np[:, -2, 0]
throttle_prev = throttle_prev[:,np.newaxis]
steering_prev = test_in_dataset_np[:, -2, 1]
steering_prev = steering_prev[:,np.newaxis]
control_actions = np.hstack((throttle, steering, throttle_prev, steering_prev))
# dX = differential_mpc_model(states, control_actions, p).T
dX = planar_model_TV(np.hstack((states,vx_prev,vy_prev)),control_actions,p,h).T

states_next_model = euler_step(states, dX, h)

for i, model in enumerate(models):
    NN_correction = model(test_in_dataset)
    states_next_NN[i] = euler_step(states, dX+NN_correction.detach().numpy(), h)
    
# States Next
states_next = test_out_dataset.numpy()*h + states_next_model

In [ ]:
fig, ax = plt.subplots(3, 1)

ax[0].plot(states_next[:,0], label='True',linewidth=0.1)
ax[0].plot(states_next_model[:,0], label='Model Predicted',linewidth=0.1, linestyle='--')

ax[1].plot(states_next[:,1], label='True',linewidth=0.1)
ax[1].plot(states_next_model[:,1], label='Model Predicted',linewidth=0.1, linestyle='--')

ax[2].plot(states_next[:,2], label='True',linewidth=0.1)
ax[2].plot(states_next_model[:,2], label='Model Predicted',linewidth=0.1, linestyle='--')

ax[0].plot(states_next_NN[0,:,0], label='NN Raw',linewidth=0.1, linestyle='--')

ax[1].plot(states_next_NN[0,:,1], label='NN Raw',linewidth=0.1, linestyle='--')

ax[2].plot(states_next_NN[0,:,2], label='NN Raw',linewidth=0.1, linestyle='--')

for i in range(1,len(models)):
    ax[0].plot(states_next_NN[i,:,0], label='NN' + str(i),linewidth=0.1, linestyle=':')

    ax[1].plot(states_next_NN[i,:,1], label='NN' + str(i),linewidth=0.1, linestyle=':')

    ax[2].plot(states_next_NN[i,:,2], label='NN' + str(i),linewidth=0.1, linestyle=':')


ax[0].set_xlabel('Timestep')
ax[0].set_ylabel('Velocity x [m/s]')
ax[0].set_title('vx')
ax[0].legend(fontsize=2)

ax[1].set_xlabel('Timestep')
ax[1].set_ylabel('Velocity y [m/s]')
ax[1].set_title('vy')
ax[1].legend(fontsize=2)

ax[2].set_xlabel('Timestep')
ax[2].set_ylabel('Yaw Rate [rad/s]')
ax[2].set_title('yaw rate',)
ax[2].legend(fontsize=2)

plt.savefig("../saved_images/NN_comparison_predict.svg", format='svg')

In [ ]:
error_model = states_next - states_next_model
error_base_NN = states_next - states_next_NN[0,:,:]
error_online_NN1 = states_next - states_next_NN[1,:,:]

col_labels = ['Vx', 'Vy', 'r']
row_labels = ['Model', 'Base NN', 'Online NN']

rmse_vx_model = np.sqrt(np.mean(error_model[:,0]**2))
rmse_vy_model = np.sqrt(np.mean(error_model[:,1]**2))
rmse_r_model = np.sqrt(np.mean(error_model[:,2]**2))
rmse_vx_base_NN = np.sqrt(np.mean(error_base_NN[:,0]**2))
rmse_vy_base_NN = np.sqrt(np.mean(error_base_NN[:,1]**2))
rmse_r_base_NN = np.sqrt(np.mean(error_base_NN[:,2]**2))
rmse_vx_online_NN1 = np.sqrt(np.mean(error_online_NN1[:,0]**2))
rmse_vy_online_NN1 = np.sqrt(np.mean(error_online_NN1[:,1]**2))
rmse_r_online_NN1 = np.sqrt(np.mean(error_online_NN1[:,2]**2))

rmse_data = [
    [np.sqrt(np.mean(error_model[:, 0]**2)), np.sqrt(np.mean(error_model[:, 1]**2)), np.sqrt(np.mean(error_model[:, 2]**2))],
    [np.sqrt(np.mean(error_base_NN[:, 0]**2)), np.sqrt(np.mean(error_base_NN[:, 1]**2)), np.sqrt(np.mean(error_base_NN[:, 2]**2))],
    [np.sqrt(np.mean(error_online_NN1[:, 0]**2)), np.sqrt(np.mean(error_online_NN1[:, 1]**2)), np.sqrt(np.mean(error_online_NN1[:, 2]**2))]
]

pd.DataFrame(rmse_data, columns=col_labels, index=row_labels)